In [3]:
# Consolidación de Datos con Python

import os
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows


def cargar_y_limpiar_datos(path_archivo: str) -> pd.DataFrame:
    """
    Carga el dataset de Service Centers desde el archivo Excel original (ml_excl),
    realiza la limpieza de datos manejando valores nulos y formatos de fecha.
    """
    if not os.path.exists(path_archivo):
        raise FileNotFoundError(f"No se encontró el archivo en la ruta: {path_archivo}")
        
    df = pd.read_excel(path_archivo)
    df.columns = df.columns.str.strip().str.lower()
    
    # AJUSTE: Mapeo de los nombres reales de tu archivo Excel
    columnas_porcentaje = ['avance_obra_civil_pct', 'avance_compras_pct', 'avance_licencias_pct']
    for col in columnas_porcentaje:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            # Si vienen en base 100 (ej: 85), los pasamos a decimal (0.85) para el formato numérico de Excel
            if df[col].max() > 1.0:
                df[col] = df[col] / 100.0
        else:
            # Por si acaso falta la columna entera, la creamos en 0 para que no truene el promedio
            df[col] = 0.0

    # Convertir fecha de apertura a datetime de forma segura
    if 'fecha_estimada_apertura' in df.columns:
        df['fecha_estimada_apertura'] = pd.to_datetime(df['fecha_estimada_apertura'], errors='coerce')
    
    return df

def calcular_metricas_consolidadas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el avance promedio total y los días de retraso acumulados.
    """
    # 1. % Avance Consolidado usando los nombres reales de tus columnas
    columnas_porcentaje = ['avance_obra_civil_pct', 'avance_compras_pct', 'avance_licencias_pct']
    df['avance total'] = df[columnas_porcentaje].mean(axis=1)
    
    # 2. Días de retraso acumulado (Si el archivo ya trae la columna 'dias_retraso', la respetamos o recalculamos)
    if 'dias_retraso' in df.columns:
        df['dias_retraso'] = pd.to_numeric(df['dias_retraso'], errors='coerce').fillna(0).astype(int)
    else:
        # Recalcular contra hoy si no existiera
        fecha_hoy = pd.to_datetime('today').normalize()
        df['dias_retraso'] = (fecha_hoy - df['fecha_estimada_apertura']).dt.days
        df['dias_retraso'] = df['dias_retraso'].clip(lower=0)

    # LÓGICA LOGÍSTICA: Si el avance total es 100% (1.0), el retraso es 0 porque ya operan
    df.loc[df['avance total'] >= 1.0, 'dias_retraso'] = 0
    df['dias_retraso'] = df['dias_retraso'].fillna(0).astype(int)
    
    return df

def exportar_a_excel_formateado(df: pd.DataFrame, nombre_salida: str = "KPI_Set_Up_Consolidado.xlsx"):
    """
    Exporta el DataFrame a un nuevo archivo de Excel con formato profesional corporativo.
    """
    wb = Workbook()
    ws = wb.active
    ws.title = "Consolidado Set Up"
    ws.views.sheetView[0].showGridLines = True  # Mantener líneas de cuadrícula activas
    
    df_excel = df.copy()
    
    # MAPEO EXACTO: De nombres de la base a nombres ejecutivos de reporte
    columnas_finales = {
        'id_sc': 'ID SC',
        'nombre_sc': 'Nombre SC',
        'estado': 'Estado',
        'tipo_sc': 'Tipo SC',
        'estatus_general': 'Fase Gantt',
        'avance_obra_civil_pct': '% Obra Civil',
        'avance_compras_pct': '% Compras',
        'avance_licencias_pct': '% Licencias',
        'fecha_estimada_apertura': 'Fecha Estimada Apertura',
        'avance total': '% Avance Total',
        'dias_retraso': 'Días de Retraso'
    }
    
    df_excel = df_excel.rename(columns=columnas_finales)
    columnas_orden = [c for c in columnas_finales.values() if c in df_excel.columns]
    df_excel = df_excel[columnas_orden]
    
    # Formatear la fecha a string estándar YYYY-MM-DD
    if 'Fecha Estimada Apertura' in df_excel.columns:
        df_excel['Fecha Estimada Apertura'] = df_excel['Fecha Estimada Apertura'].dt.strftime('%Y-%m-%d')

    # Escribir datos
    for r in dataframe_to_rows(df_excel, index=False, header=True):
        ws.append(r)
        
    # --- DISEÑO DE CORTE CORPORATIVO ---
    font_header = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
    fill_header = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid') # Azul Marino
    font_body = Font(name='Segoe UI', size=10)
    align_center = Alignment(horizontal='center', vertical='center')
    align_left = Alignment(horizontal='left', vertical='center')
    border_side = Side(border_style="thin", color="D9D9D9")
    border_cell = Border(left=border_side, right=border_side, top=border_side, bottom=border_side)
    
    # Aplicar estilos a encabezados
    for cell in ws[1]:
        cell.font = font_header
        cell.fill = fill_header
        cell.alignment = align_center
        ws.row_dimensions[1].height = 26

    # Aplicar formatos a las celdas de datos
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.font = font_body
            cell.border = border_cell
            
            header_val = ws.cell(row=1, column=cell.column).value
            
            if "%" in str(header_val):
                cell.number_format = '0.0%'
                cell.alignment = align_center
            elif "Días" in str(header_val) or "ID" in str(header_val):
                cell.number_format = '#,##0'
                cell.alignment = align_center
            elif "Fecha" in str(header_val):
                cell.alignment = align_center
            else:
                cell.alignment = align_left

    # Auto-ajustar ancho de columnas
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = col[0].column_letter
        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)
        
    wb.save(nombre_salida)
    print(f"Archivo generado correctamente: {nombre_salida}")

if __name__ == "__main__":
    # Tu ruta para guardar el excel
    archivo_input = r"C:\Users\samue\OneDrive\Escritorio\mercado libre\ml_excl.xlsx"
    
    print("Iniciando proceso ETL para el equipo de Set Up...")
    df_limpio = cargar_y_limpiar_datos(archivo_input)
    df_consolidado = calcular_metricas_consolidadas(df_limpio)
    ruta_salida = r"C:\Users\samue\OneDrive\Escritorio\mercado libre\KPI_Set_Up_Consolidado.xlsx"
    exportar_a_excel_formateado(df_consolidado, nombre_salida=ruta_salida)
    print("ETL Finalizado sin errores.")

Iniciando proceso ETL para el equipo de Set Up...
¡Éxito total! Archivo generado correctamente: C:\Users\samue\OneDrive\Escritorio\mercado libre\KPI_Set_Up_Consolidado.xlsx
ETL Finalizado sin errores.
